# Optimize MoveNet to TensorRT

**Run on Jetson Nano** (engine files are GPU-architecture-specific — a Nano engine
won't load on a PC and vice-versa).

**Goal**: turn `pose_raw.onnx` into a `.engine` file that runs ~5-10× faster
than vanilla PyTorch on the same hardware, and verify the speedup.

## What TensorRT actually does for you

Internally, the engine builder applies a stack of optimizations:

1. **Layer fusion** — `Conv + BN + ReLU` becomes one CUDA kernel, saving memory R/W
2. **Precision reduction** — FP32 → FP16 doubles throughput on Nano with negligible accuracy loss
3. **Kernel autotuning** — for each op, TRT benchmarks several CUDA implementations and picks the fastest for *this exact GPU*
4. **Memory reuse** — intermediate tensors share buffers when their lifetimes don't overlap
5. **Dead op elimination** — Dropout, training-only BN stats, etc. are stripped

Note: Maxwell GPU (Nano) does NOT support INT8 — so FP16 is the lowest precision
you can use here. INT8 is a Xavier/Orin feature.

## Steps

| Step | What |
|---|---|
| 0 | Environment + ONNX sanity check |
| 1 | Build the FP16 engine |
| 2 | Inspect the engine (inputs, outputs, sizes) |
| 3 | Benchmark TRT inference latency |
| 4 | (Optional) Compare against PyTorch baseline |
| 5 | (Optional) Build an FP32 engine as accuracy reference |

## Step 0: Environment + ONNX sanity check

Confirm TensorRT and PyCUDA are available, and the ONNX file is in place.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # noqa
import numpy as np

print("Python  :", sys.version.split()[0])
print("TensorRT:", trt.__version__)
print("NumPy   :", np.__version__)

# GPU info
dev = cuda.Device(0)
print("\nGPU     :", dev.name())
print("  total memory: {:.1f} GB".format(
    dev.total_memory() / (1024**3)))
print("  compute cap : {}.{}".format(*dev.compute_capability()))

# Maxwell = compute capability 5.x, supports FP16 but NOT INT8.
cc_major, _ = dev.compute_capability()
if cc_major == 5:
    print("  -> Maxwell architecture (Jetson Nano). FP16 OK, INT8 NOT supported.")
elif cc_major >= 7:
    print("  -> Volta or newer. INT8 also available.")

Python  : 3.6.9
TensorRT: 8.2.1.8
NumPy   : 1.19.5

GPU     : NVIDIA Tegra X1
  total memory: 1.9 GB
  compute cap : 5.3
  -> Maxwell architecture (Jetson Nano). FP16 OK, INT8 NOT supported.


In [2]:
# ============== Paths ==============
ONNX_PATH        = "output/pose_raw.onnx"     # input
ENGINE_FP16_PATH = "output/pose_fp16.engine"  # main output
# ENGINE_FP32_PATH = "output/pose_fp32.engine"  # optional baseline
# ===================================

if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(
        "ONNX file not found: {}\n"
        "Run pth2onnx.py on PC first, then scp it to the Nano.".format(ONNX_PATH))

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print("[OK] ONNX: {} ({:.1f} MB)".format(ONNX_PATH, size_mb))

[OK] ONNX: output/pose_raw.onnx (7.3 MB)


---
## Step 1: Build the FP16 engine

Uses `build_engine()` from `movenet/trt_builder.py`. The function:
1. Parses the ONNX with `trt.OnnxParser`
2. Configures the builder (FP16 flag + workspace size)
3. Calls `build_serialized_network()` and writes the bytes to disk

### About `workspace_gb`
TRT uses scratch memory during the *build* (kernel autotuning needs it). 1 GB is
enough for MoveNet on Nano. If you see `[ERROR] could not find any
implementation` in the build logs, raise this. If you OOM during build, lower it
or enable swap.

### About `verbose=True`
Set `verbose=True` if the build fails — it'll dump every layer's autotuning
decision. Otherwise it's noisy.

In [ ]:
from trt_builder import build_engine

build_engine(
    onnx_path    = ONNX_PATH,
    engine_path  = ENGINE_FP16_PATH,
    fp16         = True,
    workspace_gb = 1,
    rebuild      = False,    # True forces a rebuild even if the file exists
    verbose      = False,
)

[INFO] ONNX  : output/pose_raw.onnx (7.3 MB)
[INFO] Engine: output/pose_fp16.engine
[INFO] Config: FP16=True, workspace=1 GB
[INFO] TensorRT 8.2.1.8
[INFO] Building engine (5-15 min on Nano)...
[OK] ONNX parsed
  Input  : input (1, 3, 192, 192)
  Output : heatmap (1, 17, 48, 48)
  Output : center (1, 1, 48, 48)
  Output : regs (1, 34, 48, 48)
  Output : offsets (1, 34, 48, 48)
[OK] FP16 enabled


If the cell above failed with OOM:
```bash
# enable 4 GB swap, then re-run
sudo fallocate -l 4G /swapfile
sudo chmod 600 /swapfile
sudo mkswap /swapfile
sudo swapon /swapfile

# also free up RAM by stopping the desktop
sudo init 3
```
Then re-run the build cell.

---
## Step 2: Inspect the engine

Load the engine and print its bindings. Useful to confirm the I/O contract
matches what the downstream code expects.

In [ ]:
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

with open(ENGINE_FP16_PATH, 'rb') as f:
    runtime = trt.Runtime(TRT_LOGGER)
    engine = runtime.deserialize_cuda_engine(f.read())

print("Engine: {} ({:.2f} MB)".format(
    ENGINE_FP16_PATH,
    os.path.getsize(ENGINE_FP16_PATH) / (1024 * 1024)))
print("Bindings:")
for i in range(engine.num_bindings):
    name = engine[i]
    shape = tuple(engine.get_binding_shape(i))
    dtype = trt.nptype(engine.get_binding_dtype(i))
    kind = "input " if engine.binding_is_input(i) else "output"
    print("  [{}] {} | {} {} | {}".format(
        i, kind, name, shape, np.dtype(dtype).name))

# Expected: 1 input (1,3,192,192) + 4 outputs (heatmap, center, regs, offsets)

---
## Step 3: Benchmark TRT inference latency

Using `MoveNetTRT` from `movenet/movenet_trt.py` (the production wrapper).
Measures end-to-end latency: preprocess → TRT forward → postprocess. This
is what you'll actually see in real-time inference.

**Why warmup?** The very first inference allocates GPU memory and JITs some
kernels — it's much slower than steady-state. Always warm up before timing.

**Why CUDA sync?** GPU calls are asynchronous. Without `cuda.Stream.synchronize()`,
Python `time.time()` returns before the GPU is actually done. Our `MoveNetTRT.infer()`
already calls `stream.synchronize()` internally before returning, so the timing
is honest.

In [ ]:
import time
import cv2
from movenet_trt import MoveNetTRT, KEYPOINT_NAMES, draw_keypoints

# Use a real image if available, else a random image
TEST_IMG = "test.jpg"
if os.path.exists(TEST_IMG):
    frame = cv2.imread(TEST_IMG)
    print("Using {}, shape={}".format(TEST_IMG, frame.shape))
else:
    frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    print("No test.jpg, using random 480x640 image")

# Load engine
movenet_fp16 = MoveNetTRT(ENGINE_FP16_PATH, img_size=192)

In [ ]:
def benchmark(model, frame, n_warmup=10, n_iter=100, label=""):
    # warmup (first calls are slow due to allocation/JIT)
    for _ in range(n_warmup):
        model.infer(frame)

    # timed runs
    times = []
    for _ in range(n_iter):
        t0 = time.time()
        model.infer(frame)
        times.append(time.time() - t0)
    times = np.array(times) * 1000   # to ms

    print("\n=== {} ===".format(label))
    print("  iters: {}".format(n_iter))
    print("  mean : {:.2f} ms  ({:5.1f} FPS)".format(times.mean(), 1000 / times.mean()))
    print("  p50  : {:.2f} ms".format(np.percentile(times, 50)))
    print("  p90  : {:.2f} ms".format(np.percentile(times, 90)))
    print("  p99  : {:.2f} ms".format(np.percentile(times, 99)))
    print("  min  : {:.2f} ms".format(times.min()))
    print("  max  : {:.2f} ms".format(times.max()))
    return times

fp16_times = benchmark(movenet_fp16, frame, label="TRT FP16")

**Expected on Jetson Nano** (after `sudo nvpmodel -m 0 && sudo jetson_clocks`):
- Mean latency: 20-35 ms
- FPS: 30-50

If you're seeing < 20 FPS, run these on the Nano shell first:
```bash
sudo nvpmodel -m 0     # unlock all 4 CPU cores
sudo jetson_clocks     # raise GPU/CPU clocks to max
```
Without these, throughput drops by 2-3×.

### Sanity check — keypoint output makes sense

Latency is fine but useless if the engine outputs garbage. Quickly verify
the (17, 3) keypoints land on a plausible body.

In [ ]:
kpts = movenet_fp16.infer(frame)
print("output shape: {}, dtype: {}".format(kpts.shape, kpts.dtype))
print("\nkeypoints:")
for k, (x, y, s) in enumerate(kpts):
    print("  {:2d} {:14s}  ({:6.1f}, {:6.1f})  score={:.3f}".format(
        k, KEYPOINT_NAMES[k], x, y, s))

# If the input was a real image, save the visualization for visual inspection.
if os.path.exists(TEST_IMG):
    vis = draw_keypoints(frame, kpts)
    cv2.imwrite("trt_test_out.jpg", vis)
    print("\nVisualization saved: trt_test_out.jpg")

---
## Step 4 (optional): Compare against PyTorch baseline

Skip this if you don't have the original `.pth` on the Nano — the comparison
is mostly to *prove* TRT helps. If you trust me, the speedup is ~5-10× on
Nano.

If you do want to run it: load the `.pth`, do the same preprocess +
postprocess as `MoveNetTRT`, and time it. Below is a sketch — adapt the
model loading to whatever fire717's predict.py does on your install.

In [ ]:
# ============================================================
# SET TO None to skip this comparison.
# Otherwise point at your fire717 .pth file (the one you used
# to generate the ONNX).
# ============================================================
PTH_PATH = None    # e.g. "movenet/output/movenet.pth"

if PTH_PATH is None or not os.path.exists(PTH_PATH or ""):
    print("[SKIP] PyTorch baseline comparison")
    print("       Set PTH_PATH above to enable it.")
else:
    import torch
    # NOTE: this depends on fire717's model class. The cleanest way is to
    # import it from movenet/lib/. Adapt the import to your repo:
    #
    #   sys.path.insert(0, os.path.abspath('movenet'))
    #   from lib.movenet_mobilenetv2 import MoveNet
    #   model = MoveNet(num_classes=17, ft_size=48)
    #   model.load_state_dict(torch.load(PTH_PATH, map_location='cuda'))
    #   model = model.cuda().eval()
    #
    # Once `model` is loaded, re-implement preprocess/postprocess (or
    # reuse fire717's predict.py logic) and benchmark with the same
    # `benchmark()` helper above.
    print("PyTorch baseline scaffolding — fill in the model loading for your repo.")
    print("Expected result on Nano: ~5-10x slower than TRT FP16.")

---
## Step 5 (optional): Build an FP32 engine

Why bother? Two reasons:
1. **Accuracy reference** — if FP16 keypoints look slightly off, comparing
   to FP32 tells you whether it's a quantization issue or a deeper bug.
2. **A/B speed comparison** — see how much FP16 actually buys you on this GPU.

FP32 engine is roughly 2× larger and 1.5-2× slower on Nano.

In [ ]:
BUILD_FP32 = False    # flip to True to actually build it (takes another 5-15 min)

if BUILD_FP32:
    build_engine(
        onnx_path    = ONNX_PATH,
        engine_path  = ENGINE_FP32_PATH,
        fp16         = False,
        workspace_gb = 1,
        rebuild      = False,
    )
    movenet_fp32 = MoveNetTRT(ENGINE_FP32_PATH, img_size=192)
    fp32_times = benchmark(movenet_fp32, frame, label="TRT FP32")

    speedup = fp32_times.mean() / fp16_times.mean()
    print("\nFP16 vs FP32 speedup: {:.2f}x".format(speedup))

    # Compare keypoint outputs
    k_fp16 = movenet_fp16.infer(frame)
    k_fp32 = movenet_fp32.infer(frame)
    diff = np.abs(k_fp16[:, :2] - k_fp32[:, :2])
    print("\nFP16 vs FP32 keypoint position diff (pixels):")
    print("  mean: {:.3f}, max: {:.3f}".format(diff.mean(), diff.max()))
    print("  (sub-pixel diff is normal; should be < 1 pixel typically)")
else:
    print("[SKIP] FP32 baseline. Set BUILD_FP32=True to enable.")

---
## What's next

You now have `movenet/output/pose_fp16.engine`. Use it from `pipeline.ipynb`:

```python
from movenet.movenet_trt import MoveNetTRT
movenet = MoveNetTRT("movenet/output/pose_fp16.engine")
kpts = movenet.infer(frame)   # (17, 3) float32
```

## FAQ

**Q: Why isn't BGR→RGB part of the engine?**
It's a one-line `cv2.cvtColor` in CPU preprocess (< 1 ms). Baking it into the
ONNX would force a specific input convention on every consumer of the model,
and would require re-exporting the ONNX every time you want to test BGR vs RGB.
Standard practice: ONNX/engine stays format-agnostic, Python preprocess does
the conversion.

**Q: Engine works on PC but not on Nano (or vice versa)?**
Engines are GPU-architecture-specific. Build on the device you'll deploy on.
ONNX is portable, `.engine` is not.

**Q: Why does building take 5-15 minutes?**
Kernel autotuning. TRT runs each op with several CUDA implementations and
times them on *your* GPU. The cost is amortized — you build once, then load
the engine in < 1 second every subsequent run.

**Q: Can I use INT8 for more speed?**
Not on Nano. Maxwell GPU lacks INT8 tensor instructions. FP16 is the floor here.
On Xavier/Orin you can — but it requires a calibration dataset, not just a flag.

**Q: My FPS is way below the expected 30-50.**
Check, in order:
1. `sudo nvpmodel -m 0 && sudo jetson_clocks` — biggest single factor (2-3×)
2. Kill the desktop (`sudo init 3`) and any background services
3. Make sure you're using FP16 (re-check Step 1 logs for "FP16 enabled")
4. Confirm webcam isn't bottlenecking (test with a video file first)